In [1]:
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma
import json
from langchain_classic.schema import Document

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12984\3814599612.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
d:\Sindhu-RAG\sindhu_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open(r"../Data/processed/sindhu_report_chunks.json", "r") as file:
    data = json.load(file)

sindh_chunks = [
    Document(
        page_content=chunk["page_content"],
        metadata=chunk["metadata"]
    )
    for chunk in data
]
print(sindh_chunks)

[Document(metadata={'producer': 'Recoded by LuraDocument PDF v2.28', 'creator': 'Digitized by the Internet Archive', 'creationdate': '2011-09-21T20:43:05+00:00', 'title': 'The Cambridge history of India', 'keywords': 'http://www.archive.org/details/cambridgehistory00raps', 'author': 'Rapson, E. J. (Edward James), 1861-1937, ed', 'moddate': "D:20110921212756Z00'00'", 'source': 'D:\\Sindhu-RAG\\Data\\raw\\report.pdf', 'total_pages': 204, 'page': 0, 'page_label': '1'}, page_content='SIRMORTIMERWHEELER\nTHEINDUS\nCIVILIZATION\nTHIRD\nEDITION\ns.e>a.net$2.9!\nINU.K.INU.S.A'), Document(metadata={'producer': 'Recoded by LuraDocument PDF v2.28', 'creator': 'Digitized by the Internet Archive', 'creationdate': '2011-09-21T20:43:05+00:00', 'title': 'The Cambridge history of India', 'keywords': 'http://www.archive.org/details/cambridgehistory00raps', 'author': 'Rapson, E. J. (Edward James), 1861-1937, ed', 'moddate': "D:20110921212756Z00'00'", 'source': 'D:\\Sindhu-RAG\\Data\\raw\\report.pdf', 'to

In [3]:
embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')

vectore_store=Chroma.from_documents(
    sindh_chunks,
    embedding_model,
    collection_name="sindh-10k-collection",
    persist_directory=r"../Storage/sindhu_db"
)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12984\1973440645.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')


In [4]:
vectore_store.persist()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12984\3065356983.py:1: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectore_store.persist()


In [5]:
vectorestore_persisted=Chroma(
    collection_name="sindh-10k-collection",
    persist_directory=r"..\Storage\sindhu_db",
    embedding_function=embedding_model
)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_12984\2205999940.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorestore_persisted=Chroma(


In [6]:
from pprint import pprint
query="Who wrote indus valley civilization"

retreiver=vectorestore_persisted.as_retriever(
    search_type="similarity",
    search_kwargs={"k":5}
)

def retrieve_chunks(user_query, retriever):
    docs = retriever.invoke(user_query)
    return [
        {"index": i, "text": d.page_content, "metadata": d.metadata}
        for i, d in enumerate(docs, start=1)
    ]
pprint(retrieve_chunks(query,retreiver))

[{'index': 1,
  'metadata': {'author': 'Rapson, E. J. (Edward James), 1861-1937, ed',
               'creationdate': '2011-09-21T20:43:05+00:00',
               'creator': 'Digitized by the Internet Archive',
               'keywords': 'http://www.archive.org/details/cambridgehistory00raps',
               'moddate': "D:20110921212756Z00'00'",
               'page': 160,
               'page_label': '161',
               'producer': 'Recoded by LuraDocument PDF v2.28',
               'source': 'D:\\Sindhu-RAG\\Data\\raw\\report.pdf',
               'title': 'The Cambridge history of India',
               'total_pages': 204},
  'text': '1961).\n'
          '11.Fairservis,WalterA.,Jr.,TheOrigin,Character,andDeclineofanEarly\n'
          'Civilisation(AmericanMuseumNovitates,no.2302,20Oct.1967).\n'
          '12.Gordon,D.H.,ThePrehistoricBackgroundofIndianCulture(Bombay,\n'
          '1958).\n'
          "13.Khan,F.A.,'Excavations atKotDiji',inPakistanArchaeology,no.2\n"
          '(Depa